# HadCET: quickstart

Download and plot the Central England Temperature monthly mean series,
1659 to present. The longest instrumental temperature record in the world,
a single homogenised time series for a triangular area of lowland central
England.

See [`docs/hadcet/README.md`](../docs/hadcet/README.md) for the full
reference. No authentication; just `pip install pandas requests matplotlib`.

In [ ]:
# ==================================================================
# USER CONFIGURATION - Edit these values for your use case
# ==================================================================
RESOLUTION = "monthly"   # "monthly" or "daily"
STATISTIC = "mean"        # "mean", "max", or "min"
OUTPUT_DIR = "../data/hadcet"
OUTPUT_FILENAME = "hadcet_monthly_mean.csv"
# ==================================================================

## Imports and environment check

In [ ]:
import sys
from importlib.metadata import version
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import requests

print(f"Python       {sys.version.split()[0]}")
for pkg in ["pandas", "matplotlib", "requests"]:
    print(f"{pkg:12} {version(pkg)}")

def _find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for parent in [here, *here.parents]:
        if (parent / "CLAUDE.md").exists():
            return parent
    raise RuntimeError("Could not find repo root.")

repo_root = _find_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

## Download and parse

HadCET publishes plain text files at stable URLs. We pull the monthly mean
file and reshape into long format so each row is one month.

In [ ]:
from scripts.hadcet_download import download, _url_for  # noqa: E402

url = _url_for(RESOLUTION, STATISTIC)
print(f"Fetching {url}")

output_path = download(
    resolution=RESOLUTION,
    statistic=STATISTIC,
    output_dir=OUTPUT_DIR,
    output_filename=OUTPUT_FILENAME,
)
print(f"Saved: {output_path}")

## Open and inspect

~4,400 monthly rows from 1659 to present.

In [ ]:
df = pd.read_csv(output_path, parse_dates=["date"])
print(df.head())
print(df.tail())
print(f"\nTotal rows: {len(df)}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

## Plot the full series

Monthly values are noisy. Overlay a 10-year rolling mean to show the
long-term signal clearly.

In [ ]:
df = df.set_index("date")

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df.index, df["temperature_degC"], color="lightgray", linewidth=0.5, label="Monthly mean")
rolling = df["temperature_degC"].rolling(window=120, center=True).mean()
ax.plot(rolling.index, rolling, color="C3", linewidth=1.5, label="10-year rolling mean")

ax.set_xlabel("Year")
ax.set_ylabel("Temperature (C)")
ax.set_title("HadCET monthly mean temperature, 1659 to present")
ax.grid(alpha=0.3)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

## Next steps

- **Full reference**: [`docs/hadcet/README.md`](../docs/hadcet/README.md)
- **Daily series**: set `RESOLUTION = "daily"` in the config cell.
  Available from 1772 for mean, 1878 for max/min.
- **Compare with ERA5**: overlay HadCET against ERA5 single-levels
  temperature for central England to validate reanalysis skill across the
  backextension period.
- **UK-wide gridded record**: for UK national averages rather than central
  England, see the Met Office HadUK-Grid product (not in this repo).